# Sharing artifacts through cloud storage

`FsspecArtifactStore` keeps artifacts in S3, GCS or Azure, so a model trained on one machine can be reloaded and evaluated on another. Each machine keeps a local cache and transfers only files that changed.

This notebook uses fsspec's in-memory filesystem so it runs without cloud credentials; two stores with separate caches stand in for two machines. For real storage, install your cloud's extra and change the URL:

```bash
pip install "katabatic[artifacts-s3]"   # or artifacts-gcs, artifacts-azure
```

```python
store = FsspecArtifactStore("s3://my-bucket/katabatic", local_cache_dir="artifact-cache")
```

In [ ]:
from importlib.resources import files
from pathlib import Path

import pandas as pd

from katabatic.artifacts import ArtifactConflictError, FsspecArtifactStore
from katabatic.models import ModelRegistry, get_model
from katabatic.pipeline import TrainTestSplitPipeline

car = pd.read_csv(files("katabatic.datasets") / "car.csv")
car.columns = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]
Path("artifacts").mkdir(exist_ok=True)
car.to_csv("artifacts/car.csv", index=False)

BUCKET = "memory://katabatic-demo"  # e.g. "s3://my-bucket/katabatic"

## Train on one machine

In [2]:
laptop = FsspecArtifactStore(BUCKET, local_cache_dir="artifacts/cache-laptop")
results = TrainTestSplitPipeline(model=get_model("naivebayes")).run(
    input_csv="artifacts/car.csv",
    dataset_name="car",
    artifact_store=laptop,
    model_name="naivebayes",
    target_column="class",
)
model_ref = results["model_ref"]

## Reload on another

The second store starts with an empty cache. `load_from_ref()` downloads the model's state from the bucket.

In [3]:
server = FsspecArtifactStore(BUCKET, local_cache_dir="artifacts/cache-server")
model = ModelRegistry.load_model("naivebayes").load_from_ref(server, model_ref)
model.sample(5)

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,low,5more,4,big,med,unacc
1,vhigh,vhigh,2,2,med,med,unacc
2,med,vhigh,3,2,med,high,unacc
3,vhigh,med,4,2,small,med,unacc
4,med,high,3,4,big,med,acc


## Conflicting writes

A store never overwrites a file that changed remotely since it last read it. `save_json()`, `save_bytes()` and `sync()` raise `ArtifactConflictError` instead: load the file again, re-apply your change, and save.

In [4]:
laptop.save_json("notes.json", {"owner": "laptop"})
server.load_json("notes.json")

laptop.save_json("notes.json", {"owner": "laptop", "status": "ready"})
try:
    server.save_json("notes.json", {"owner": "server"})  # based on the old version
except ArtifactConflictError as err:
    conflict = err
conflict

katabatic.artifacts.base.ArtifactConflictError('notes.json changed remotely since this store last read it; load it, re-apply your change, and save again.')

In [5]:
notes = server.load_json("notes.json")
notes["reviewed_by"] = "server"
server.save_json("notes.json", notes)
laptop.load_json("notes.json")

{'owner': 'laptop', 'status': 'ready', 'reviewed_by': 'server'}

## Before you use it

- **Only load models from storage you trust.** Most models save their state with pickle, so loading a model runs code from the bucket.
- A store is thread-safe, but don't share one `local_cache_dir` between processes.
- `exists()` downloads the file it checks, so `open_path()` can read it afterwards.